# Invoice Automation AI Agent

## AI Legends 2026 — AI Agent Automation Track

Энэ notebook нь invoice зураг болон PDF файлуудаас мэдээлэл олборлож, master database-тай тулган шалгаж, эрсдэл илрүүлж, санхүүгийн ангилал оноож, эцсийн бизнес шийдвэр гаргадаг AI agent pipeline юм.

**Гол боломжууд:**

- PDF/JPG/PNG invoice файлыг автоматаар илрүүлэх
- Groq Vision model ашиглан structured field extraction хийх
- Vendor, bank account, amount, date, duplicate validation хийх
- `AUTO_POST`, `HUMAN_APPROVAL`, `DENY` шийдвэр гаргах
- Нийт үр дүн дээр chatbot-style Q&A хийх
- Optional Gradio interface ашиглан demo хийх

> Энэ notebook нь шинэ dataset нэмэгдсэн үед input folder-ийг hardcode хийхгүйгээр автоматаар scan хийхээр бүтээгдсэн.

## 1. Competition Requirement Mapping

| Competition requirement | Notebook section |
|---|---|
| Extract information from invoice image/PDF | Vision-based extraction |
| Classify invoice into financial category | Category classification |
| Detect errors and risks | Validation + risk flagging |
| Make final decision | Final decision logic |
| Answer aggregate questions | Chatbot Q&A agent |
| Public notebook reproducibility | Clear config, outputs, dependency list |

**Required risk types:**

- `AMOUNT_MISMATCH`
- `UNREGISTERED_VENDOR`
- `INVALID_DATE`
- `BANK_ACCOUNT_MISMATCH`
- `DUPLICATE`

**Required final decisions:**

- `AUTO_POST`
- `HUMAN_APPROVAL`
- `DENY`

## 2. Install Dependencies

Kaggle runtime дээр зарим library байхгүй байж болно. Энэ cell нь шаардлагатай package-уудыг суулгана.

In [1]:
!pip install -q groq pymupdf pillow pandas numpy rapidfuzz gradio



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Import Libraries

In [ ]:
import os
import re
import io
import json
import time
import base64
import sqlite3
import traceback
from pathlib import Path
from datetime import datetime, date
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from rapidfuzz import fuzz, process

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from groq import Groq
except Exception:
    Groq = None

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

print('Сангууд амжилттай import хийгдлээ.')

## 4. Configuration

Энэ хэсэгт input/output path, model name, processing limit зэргийг тохируулна.

In [ ]:
# Kaggle competition dataset root. Шинэ dataset нэмэгдсэн ч энэ folder дотор scan хийнэ.
DEFAULT_INPUT_ROOT = Path('/kaggle/input')
COMPETITION_NAME = 'ai-legends-2026-ai-agents-automation'
COMPETITION_DIR = DEFAULT_INPUT_ROOT / 'competitions' / COMPETITION_NAME

# Kaggle working output folder
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Vision model. Groq дээр боломжтой vision model ашиглана.
GROQ_VISION_MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Debug үед эхний хэдэн invoice боловсруулах. None бол бүх invoice.
MAX_FILES_TO_PROCESS = None

# PDF-ийн эхний хэдэн page унших. Invoice ихэнхдээ 1 page байдаг.
MAX_PDF_PAGES = 2

print('Input root:', DEFAULT_INPUT_ROOT)
print('Competition dir:', COMPETITION_DIR)
print('Output dir:', OUTPUT_DIR)

## 5. Load Multiple Groq API Keys

Kaggle Secrets дээр дараах нэрүүдээр key хадгалж болно:

- `GROQ_API_KEY_1`
- `GROQ_API_KEY_2`
- `GROQ_API_KEY_3`
- `GROQ_API_KEY_4`
- `GROQ_API_KEY_5`

Fallback байдлаар `GROQ_API_KEY` болон `API` нэрийг мөн шалгана. API key-г notebook дотор шууд бичихгүй.

In [ ]:
def load_groq_api_keys() -> List[str]:
    """Load multiple Groq API keys from Kaggle Secrets or environment variables."""
    keys = []
    secret_names = [f'GROQ_API_KEY_{i}' for i in range(1, 6)] + ['GROQ_API_KEY', 'API']

    # Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        for name in secret_names:
            try:
                value = user_secrets.get_secret(name)
                if value and value not in keys:
                    keys.append(value)
            except Exception:
                pass
    except Exception:
        pass

    # Environment variables fallback
    for name in secret_names:
        value = os.environ.get(name)
        if value and value not in keys:
            keys.append(value)

    return keys

GROQ_API_KEYS = load_groq_api_keys()
print(f'Loaded {len(GROQ_API_KEYS)} Groq API key(s).')

if len(GROQ_API_KEYS) == 0:
    print('Анхааруулга: Groq API key олдсонгүй. Vision extraction ажиллахгүй, fallback demo mode ашиглагдана.')

## 6. Auto-detect Dataset Files

Энэ хэсэг шинэ data ирсэн үед бүх PDF/JPG/PNG invoice болон database файлыг автоматаар олно.

In [ ]:
def find_existing_root() -> Path:
    """Choose the most likely dataset root."""
    if COMPETITION_DIR.exists():
        return COMPETITION_DIR
    if DEFAULT_INPUT_ROOT.exists():
        return DEFAULT_INPUT_ROOT
    return Path('.')

DATA_ROOT = find_existing_root()
print('Selected DATA_ROOT:', DATA_ROOT)


def scan_invoice_files(root: Path) -> List[Path]:
    patterns = ['*.jpg', '*.jpeg', '*.png', '*.pdf']
    files = []
    for pattern in patterns:
        files.extend(root.rglob(pattern))
    # exclude output/generated folders if running locally
    files = [p for p in files if 'outputs' not in str(p).lower()]
    return sorted(files)


def scan_database_files(root: Path) -> List[Path]:
    patterns = ['*.db', '*.sqlite', '*.sqlite3', '*.csv', '*.xlsx']
    files = []
    for pattern in patterns:
        files.extend(root.rglob(pattern))
    return sorted(files)

invoice_files = scan_invoice_files(DATA_ROOT)
database_files = scan_database_files(DATA_ROOT)

if MAX_FILES_TO_PROCESS:
    invoice_files = invoice_files[:MAX_FILES_TO_PROCESS]

print(f'Олдсон invoice файлын тоо: {len(invoice_files)}')
print(f'Олдсон database/data файлын тоо: {len(database_files)}')

pd.DataFrame({'invoice_file': [str(p) for p in invoice_files[:20]]})

## 7. Load Master Database

Master database нь vendor, category, historical invoice зэрэг structured мэдээллийг агуулна. Энэ хэсэг SQLite database-г автоматаар уншиж, table бүрийг DataFrame болгоно.

In [ ]:
def load_sqlite_database(db_path: Path) -> Dict[str, pd.DataFrame]:
    tables = {}
    try:
        conn = sqlite3.connect(db_path)
        table_names = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)['name'].tolist()
        for table in table_names:
            try:
                tables[table] = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)
            except Exception as e:
                print(f'{table} унших үед алдаа: {e}')
        conn.close()
    except Exception as e:
        print('SQLite database унших үед алдаа:', e)
    return tables

sqlite_files = [p for p in database_files if p.suffix.lower() in ['.db', '.sqlite', '.sqlite3']]
MASTER_DB_PATH = sqlite_files[0] if sqlite_files else None

master_tables = {}
if MASTER_DB_PATH:
    print('Master database:', MASTER_DB_PATH)
    master_tables = load_sqlite_database(MASTER_DB_PATH)
    print('Tables:', list(master_tables.keys()))
else:
    print('Master SQLite database олдсонгүй.')

for name, df in master_tables.items():
    print(f'\n{name}: shape={df.shape}')
    display(df.head())

## 8. Master Data Helpers

Column нэр өөр байсан ч аль болох уян хатан ажиллах helper функцууд.

In [ ]:
def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ''
    text = str(value).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def normalize_number(value: Any) -> Optional[float]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value)
    text = text.replace(',', '').replace('₮', '').replace('mnt', '').replace('MNT', '')
    text = re.sub(r'[^0-9.\-]', '', text)
    if text in ['', '.', '-', '-.']:
        return None
    try:
        return float(text)
    except Exception:
        return None


def find_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in df.columns:
        c_norm = c.lower().replace('_', '').replace(' ', '')
        for cand in candidates:
            cand_norm = cand.lower().replace('_', '').replace(' ', '')
            if cand_norm in c_norm or c_norm in cand_norm:
                return c
    return None


def get_table(possible_names: List[str]) -> Optional[pd.DataFrame]:
    lower_map = {name.lower(): name for name in master_tables.keys()}
    for name in possible_names:
        if name.lower() in lower_map:
            return master_tables[lower_map[name.lower()]]
    # fuzzy partial match
    for table_name, df in master_tables.items():
        for name in possible_names:
            if name.lower() in table_name.lower():
                return df
    return None

vendors_df = get_table(['Vendors', 'Vendor', 'Suppliers'])
items_df = get_table(['Items', 'Item'])
categories_df = get_table(['InvoiceCategories', 'Categories', 'Category'])
historical_invoices_df = get_table(['Invoices', 'Invoice', 'HistoricalInvoices'])

print('vendors_df:', None if vendors_df is None else vendors_df.shape)
print('items_df:', None if items_df is None else items_df.shape)
print('categories_df:', None if categories_df is None else categories_df.shape)
print('historical_invoices_df:', None if historical_invoices_df is None else historical_invoices_df.shape)

## 9. Image/PDF Conversion Helpers

PDF invoice-ийг image болгож Vision model-д дамжуулна.

In [ ]:
def image_to_data_url(image: Image.Image, max_size: int = 1600) -> str:
    """Convert PIL image to base64 data URL."""
    image = image.convert('RGB')
    w, h = image.size
    scale = min(max_size / max(w, h), 1.0)
    if scale < 1.0:
        image = image.resize((int(w * scale), int(h * scale)))

    buffer = io.BytesIO()
    image.save(buffer, format='JPEG', quality=90)
    b64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
    return f'data:image/jpeg;base64,{b64}'


def file_to_images(file_path: Path, max_pdf_pages: int = MAX_PDF_PAGES) -> List[Image.Image]:
    """Load image file or convert PDF pages to PIL images."""
    suffix = file_path.suffix.lower()
    images = []

    if suffix in ['.jpg', '.jpeg', '.png']:
        images.append(Image.open(file_path).convert('RGB'))
        return images

    if suffix == '.pdf':
        if fitz is None:
            raise RuntimeError('PyMuPDF байхгүй тул PDF унших боломжгүй.')
        doc = fitz.open(str(file_path))
        for page_index in range(min(len(doc), max_pdf_pages)):
            page = doc[page_index]
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
            img = Image.open(io.BytesIO(pix.tobytes('png'))).convert('RGB')
            images.append(img)
        doc.close()
        return images

    raise ValueError(f'Unsupported file type: {suffix}')

# Test local preview only if files exist
if invoice_files:
    test_images = file_to_images(invoice_files[0])
    print(f'Preview file: {invoice_files[0].name}, image pages: {len(test_images)}, first size: {test_images[0].size}')
else:
    print('Invoice file олдсонгүй.')

## 10. Vision-based Invoice Extraction

Groq Vision model invoice-оос structured JSON талбаруудыг буцаана.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """
You are an invoice information extraction agent.
Extract structured fields from Mongolian or English invoice images.
Return ONLY valid JSON. No markdown. No explanation.
If a value is missing, use null.
"""

EXTRACTION_USER_PROMPT = """
Extract the following invoice fields and return valid JSON only:

{
  "invoice_number": null,
  "vendor_name": null,
  "invoice_date": null,
  "due_date": null,
  "bank_name": null,
  "bank_account": null,
  "email": null,
  "currency": "MNT",
  "subtotal": null,
  "tax": null,
  "total_amount": null,
  "items": [
    {
      "description": null,
      "quantity": null,
      "unit_price": null,
      "line_total": null
    }
  ]
}

Rules:
- Keep vendor_name as written on the invoice.
- Convert amounts to numbers if possible.
- Dates should be ISO format YYYY-MM-DD if possible.
- For Mongolian invoices, preserve Mongolian names.
- Return JSON only.
"""


def extract_json_from_text(text: str) -> Dict[str, Any]:
    """Parse JSON object from model response."""
    if not text:
        return {}
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return {}
    return {}


def call_groq_with_fallback(messages: List[Dict[str, Any]], model: str = GROQ_VISION_MODEL, max_tokens: int = 1800) -> str:
    """Call Groq with multiple API key fallback."""
    if Groq is None:
        raise RuntimeError('groq package import хийгдээгүй байна.')
    if not GROQ_API_KEYS:
        raise RuntimeError('Groq API key олдсонгүй.')

    last_error = None
    for idx, api_key in enumerate(GROQ_API_KEYS, start=1):
        try:
            client = Groq(api_key=api_key, max_retries=1, timeout=60)
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0,
                max_completion_tokens=max_tokens,
            )
            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            print(f'Groq API key {idx} дээр алдаа гарлаа. Дараагийн key ашиглана...')
            time.sleep(2)
    raise RuntimeError(f'Бүх Groq API key амжилтгүй боллоо. Last error: {last_error}')


def extract_invoice_with_groq(file_path: Path) -> Dict[str, Any]:
    """Extract invoice fields using Groq Vision."""
    images = file_to_images(file_path)
    content = [{"type": "text", "text": EXTRACTION_USER_PROMPT}]
    for img in images:
        content.append({"type": "image_url", "image_url": {"url": image_to_data_url(img)}})

    messages = [
        {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
        {"role": "user", "content": content},
    ]
    raw = call_groq_with_fallback(messages)
    parsed = extract_json_from_text(raw)
    parsed['_raw_model_response'] = raw
    return parsed


def fallback_empty_extraction(file_path: Path) -> Dict[str, Any]:
    """Fallback when API is not available. Keeps pipeline running but marks extraction as failed."""
    return {
        'invoice_number': file_path.stem,
        'vendor_name': None,
        'invoice_date': None,
        'due_date': None,
        'bank_name': None,
        'bank_account': None,
        'email': None,
        'currency': 'MNT',
        'subtotal': None,
        'tax': None,
        'total_amount': None,
        'items': [],
        '_raw_model_response': None,
        '_fallback_mode': True,
    }

print('Vision extraction functions ready.')

## 11. Field Normalization

Model output-ийг validation хийхэд тохиромжтой structured format болгоно.

In [ ]:
def normalize_date(value: Any) -> Optional[str]:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None

    # common separators and Mongolian date markers
    text = text.replace('он', '-').replace('сар', '-').replace('өдөр', '')
    text = re.sub(r'[./]', '-', text)
    text = re.sub(r'\s+', '', text)

    candidates = [text]
    # Extract YYYY-MM-DD-like pattern
    m = re.search(r'(20\d{2}|19\d{2})[-年]?(\d{1,2})[-月]?(\d{1,2})', text)
    if m:
        candidates.insert(0, f'{m.group(1)}-{m.group(2)}-{m.group(3)}')

    for cand in candidates:
        for fmt in ['%Y-%m-%d', '%Y-%m-%d', '%d-%m-%Y', '%m-%d-%Y']:
            try:
                dt = datetime.strptime(cand, fmt).date()
                return dt.isoformat()
            except Exception:
                pass
    return None


def normalize_items(items: Any) -> List[Dict[str, Any]]:
    if not isinstance(items, list):
        return []
    normalized = []
    for item in items:
        if not isinstance(item, dict):
            continue
        q = normalize_number(item.get('quantity'))
        u = normalize_number(item.get('unit_price'))
        lt = normalize_number(item.get('line_total'))
        normalized.append({
            'description': item.get('description'),
            'quantity': q,
            'unit_price': u,
            'line_total': lt,
            'calculated_line_total': q * u if q is not None and u is not None else None,
        })
    return normalized


def normalize_invoice_data(raw: Dict[str, Any], file_path: Path) -> Dict[str, Any]:
    items = normalize_items(raw.get('items'))
    total_from_items = sum([i['calculated_line_total'] for i in items if i.get('calculated_line_total') is not None])
    total_from_items = total_from_items if total_from_items > 0 else None

    data = {
        'file_name': file_path.name,
        'file_path': str(file_path),
        'file_type': file_path.suffix.lower().replace('.', ''),
        'invoice_number': raw.get('invoice_number') or file_path.stem,
        'vendor_name': raw.get('vendor_name'),
        'invoice_date': normalize_date(raw.get('invoice_date')),
        'due_date': normalize_date(raw.get('due_date')),
        'bank_name': raw.get('bank_name'),
        'bank_account': raw.get('bank_account'),
        'email': raw.get('email'),
        'currency': raw.get('currency') or 'MNT',
        'subtotal': normalize_number(raw.get('subtotal')),
        'tax': normalize_number(raw.get('tax')),
        'total_amount': normalize_number(raw.get('total_amount')),
        'items': items,
        'items_text': ' | '.join([str(i.get('description') or '') for i in items]),
        'calculated_total_from_items': total_from_items,
        'raw_model_response': raw.get('_raw_model_response'),
        'fallback_mode': bool(raw.get('_fallback_mode', False)),
    }
    return data

print('Normalization functions ready.')

## 12. Category Classification

Category-г historical database болон keyword rule ашиглан онооно. Custom ML training хийхгүй, учир нь dataset өөрчлөгдөх боломжтой бөгөөд rule + historical matching нь илүү тайлбарлагдахуйц.

In [ ]:
CATEGORY_KEYWORDS = {
    'Software / IT': ['сервер', 'server', 'hosting', 'ssl', 'software', 'domain', 'cloud', 'api', 'license', 'лиценз'],
    'Office Supplies': ['оффис', 'бичиг', 'цаас', 'printer', 'paper', 'supplies', 'хэрэгсэл'],
    'Utilities': ['цахилгаан', 'ус', 'дулаан', 'internet', 'интернет', 'utility', 'холбоо'],
    'Travel Expense': ['томилолт', 'зочид', 'hotel', 'taxi', 'такси', 'flight', 'нислэг', 'travel'],
    'Maintenance': ['засвар', 'үйлчилгээ', 'maintenance', 'repair', 'support'],
    'Training / Consulting': ['сургалт', 'зөвлөгөө', 'consulting', 'training', 'workshop'],
}


def historical_category_match(vendor_name: Any) -> Optional[str]:
    if historical_invoices_df is None or vendor_name is None:
        return None
    vendor_col = find_column(historical_invoices_df, ['VendorName', 'Vendor', 'vendor_name', 'Name'])
    category_col = find_column(historical_invoices_df, ['Category', 'InvoiceCategory', 'category_name'])
    if not vendor_col or not category_col:
        return None

    target = normalize_text(vendor_name)
    if not target:
        return None
    temp = historical_invoices_df.copy()
    temp['_score'] = temp[vendor_col].apply(lambda x: fuzz.token_sort_ratio(target, normalize_text(x)))
    best = temp.sort_values('_score', ascending=False).head(1)
    if not best.empty and best['_score'].iloc[0] >= 85:
        return str(best[category_col].iloc[0])
    return None


def keyword_category_match(text: str) -> str:
    text_norm = normalize_text(text)
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(k.lower() in text_norm for k in keywords):
            return category
    return 'Other'


def classify_category(invoice: Dict[str, Any]) -> str:
    hist = historical_category_match(invoice.get('vendor_name'))
    if hist:
        return hist
    combined_text = ' '.join([
        str(invoice.get('vendor_name') or ''),
        str(invoice.get('items_text') or ''),
    ])
    return keyword_category_match(combined_text)

print('Category classification ready.')

## 13. Validation Rules

Энэ хэсэг competition-д шаардсан гол зөрчлүүдийг илрүүлнэ.

In [ ]:
def validate_registered_vendor(vendor_name: Any) -> Tuple[bool, Optional[str], float]:
    if vendors_df is None or vendor_name is None:
        return False, None, 0.0
    name_col = find_column(vendors_df, ['Name', 'VendorName', 'Vendor', 'vendor_name'])
    if not name_col:
        return False, None, 0.0

    choices = vendors_df[name_col].dropna().astype(str).tolist()
    if not choices:
        return False, None, 0.0

    match = process.extractOne(str(vendor_name), choices, scorer=fuzz.token_sort_ratio)
    if match and match[1] >= 80:
        return True, match[0], float(match[1])
    return False, match[0] if match else None, float(match[1]) if match else 0.0


def validate_amount(invoice: Dict[str, Any], tolerance: float = 1.0) -> bool:
    total = invoice.get('total_amount')
    calculated = invoice.get('calculated_total_from_items')
    if total is None or calculated is None:
        # Missing fields are handled separately; do not mark as mismatch by default.
        return True
    return abs(float(total) - float(calculated)) <= tolerance


def validate_date(invoice: Dict[str, Any]) -> bool:
    invoice_date = invoice.get('invoice_date')
    due_date = invoice.get('due_date')
    today = date.today()

    try:
        inv_dt = datetime.fromisoformat(invoice_date).date() if invoice_date else None
        due_dt = datetime.fromisoformat(due_date).date() if due_date else None
    except Exception:
        return False

    if inv_dt and inv_dt > today:
        return False
    if inv_dt and due_dt and due_dt < inv_dt:
        return False
    return True


def validate_bank_account(invoice: Dict[str, Any], matched_vendor_name: Optional[str]) -> bool:
    """Check bank account if Vendors table has an account-like column. If unavailable, return True."""
    if vendors_df is None or not matched_vendor_name:
        return True

    vendor_col = find_column(vendors_df, ['Name', 'VendorName', 'Vendor', 'vendor_name'])
    account_col = find_column(vendors_df, ['BankAccount', 'AccountNumber', 'Account', 'bank_account', 'Данс'])
    if not vendor_col or not account_col:
        return True

    invoice_account = re.sub(r'\D', '', str(invoice.get('bank_account') or ''))
    if not invoice_account:
        return True

    row = vendors_df[vendors_df[vendor_col].astype(str) == str(matched_vendor_name)]
    if row.empty:
        return True
    master_account = re.sub(r'\D', '', str(row.iloc[0][account_col]))
    if not master_account:
        return True
    return invoice_account == master_account


def detect_duplicate(invoice: Dict[str, Any]) -> bool:
    if historical_invoices_df is None:
        return False

    inv_num_col = find_column(historical_invoices_df, ['InvoiceNumber', 'invoice_number', 'Number', 'ID'])
    vendor_col = find_column(historical_invoices_df, ['VendorName', 'Vendor', 'vendor_name', 'Name'])
    total_col = find_column(historical_invoices_df, ['GrandTotal', 'TotalAmount', 'total_amount', 'Amount'])
    date_col = find_column(historical_invoices_df, ['InvoiceDate', 'Date', 'invoice_date'])

    inv_num = normalize_text(invoice.get('invoice_number'))
    if inv_num and inv_num_col:
        if historical_invoices_df[inv_num_col].astype(str).apply(normalize_text).eq(inv_num).any():
            return True

    # fallback duplicate: same vendor + same amount + same date
    if vendor_col and total_col:
        vendor = normalize_text(invoice.get('vendor_name'))
        total = invoice.get('total_amount')
        inv_date = invoice.get('invoice_date')
        for _, row in historical_invoices_df.iterrows():
            vendor_score = fuzz.token_sort_ratio(vendor, normalize_text(row.get(vendor_col))) if vendor else 0
            total_same = total is not None and normalize_number(row.get(total_col)) is not None and abs(total - normalize_number(row.get(total_col))) <= 1
            date_same = True
            if date_col and inv_date:
                date_same = normalize_date(row.get(date_col)) == inv_date
            if vendor_score >= 85 and total_same and date_same:
                return True
    return False

print('Validation rules ready.')

## 14. Risk Flagging and Final Decision Logic

Business decision rule:

- Serious risk → `DENY`
- New/uncertain invoice → `HUMAN_APPROVAL`
- Clean and known invoice → `AUTO_POST`

In [ ]:
def assign_risk_flags(invoice: Dict[str, Any]) -> Dict[str, Any]:
    flags = []

    registered, matched_vendor, vendor_score = validate_registered_vendor(invoice.get('vendor_name'))
    amount_ok = validate_amount(invoice)
    date_ok = validate_date(invoice)
    bank_ok = validate_bank_account(invoice, matched_vendor)
    duplicate = detect_duplicate(invoice)

    if not amount_ok:
        flags.append('AMOUNT_MISMATCH')
    if not registered:
        flags.append('UNREGISTERED_VENDOR')
    if not date_ok:
        flags.append('INVALID_DATE')
    if not bank_ok:
        flags.append('BANK_ACCOUNT_MISMATCH')
    if duplicate:
        flags.append('DUPLICATE')
    if invoice.get('fallback_mode'):
        flags.append('LOW_CONFIDENCE_EXTRACTION')

    invoice.update({
        'is_registered_vendor': registered,
        'matched_vendor_name': matched_vendor,
        'vendor_match_score': vendor_score,
        'amount_check': amount_ok,
        'date_check': date_ok,
        'bank_account_check': bank_ok,
        'duplicate_check': duplicate,
        'risk_flags': flags,
    })
    return invoice


def make_final_decision(invoice: Dict[str, Any]) -> str:
    flags = set(invoice.get('risk_flags') or [])
    deny_flags = {'AMOUNT_MISMATCH', 'INVALID_DATE', 'BANK_ACCOUNT_MISMATCH', 'DUPLICATE'}

    if flags.intersection(deny_flags):
        return 'DENY'
    if 'UNREGISTERED_VENDOR' in flags or 'LOW_CONFIDENCE_EXTRACTION' in flags:
        return 'HUMAN_APPROVAL'
    return 'AUTO_POST'


def make_explanation(invoice: Dict[str, Any]) -> str:
    flags = invoice.get('risk_flags') or []
    if not flags:
        return 'No risk detected. Vendor, date, amount and duplicate checks passed.'
    return 'Detected risk flags: ' + ', '.join(flags)

print('Risk flagging and decision logic ready.')

## 15. Single Invoice Processing Function

In [ ]:
def process_single_invoice(file_path: Path) -> Dict[str, Any]:
    start = time.time()
    result = {
        'file_name': file_path.name,
        'file_path': str(file_path),
        'file_type': file_path.suffix.lower().replace('.', ''),
        'processing_status': 'FAILED',
        'error_message': None,
    }

    try:
        if GROQ_API_KEYS:
            raw = extract_invoice_with_groq(file_path)
        else:
            raw = fallback_empty_extraction(file_path)

        invoice = normalize_invoice_data(raw, file_path)
        invoice['category'] = classify_category(invoice)
        invoice = assign_risk_flags(invoice)
        invoice['final_decision'] = make_final_decision(invoice)
        invoice['explanation'] = make_explanation(invoice)
        invoice['processing_status'] = 'SUCCESS'
        invoice['processing_time_sec'] = round(time.time() - start, 2)
        return invoice

    except Exception as e:
        result['error_message'] = str(e)
        result['processing_time_sec'] = round(time.time() - start, 2)
        return result

print('Single invoice processor ready.')

## 16. Batch Processing

Бүх invoice файлыг нэг нэгээр нь боловсруулна. Энэ нь safe sequential mode бөгөөд Kaggle дээр тогтвортой ажиллах зорилготой.

In [ ]:
processed_results = []

print(f'Боловсруулах invoice файлын тоо: {len(invoice_files)}')

for idx, file_path in enumerate(invoice_files, start=1):
    print(f'[{idx}/{len(invoice_files)}] Processing: {file_path.name}')
    result = process_single_invoice(file_path)
    processed_results.append(result)

results_df = pd.DataFrame(processed_results)

print('\n========== БОЛОВСРУУЛАЛТ ДУУСЛАА ==========', )
print('Нийт файл:', len(results_df))
if not results_df.empty:
    print('Амжилттай:', (results_df.get('processing_status') == 'SUCCESS').sum())
    print('Амжилтгүй:', (results_df.get('processing_status') == 'FAILED').sum())

display(results_df.head(10))

## 17. Export Results

Шүүлт, demo, GitHub output-д ашиглах CSV файлуудыг export хийнэ.

In [ ]:
def serialize_for_csv(value):
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value

if results_df.empty:
    print('Export хийх result алга.')
else:
    export_df = results_df.copy()
    for col in export_df.columns:
        export_df[col] = export_df[col].apply(serialize_for_csv)

    final_path = OUTPUT_DIR / 'final_invoice_results.csv'
    clean_path = OUTPUT_DIR / 'auto_post_invoices.csv'
    human_path = OUTPUT_DIR / 'human_approval_invoices.csv'
    deny_path = OUTPUT_DIR / 'denied_invoices.csv'
    failed_path = OUTPUT_DIR / 'failed_files.csv'

    export_df.to_csv(final_path, index=False, encoding='utf-8-sig')
    export_df[export_df.get('final_decision') == 'AUTO_POST'].to_csv(clean_path, index=False, encoding='utf-8-sig')
    export_df[export_df.get('final_decision') == 'HUMAN_APPROVAL'].to_csv(human_path, index=False, encoding='utf-8-sig')
    export_df[export_df.get('final_decision') == 'DENY'].to_csv(deny_path, index=False, encoding='utf-8-sig')
    export_df[export_df.get('processing_status') == 'FAILED'].to_csv(failed_path, index=False, encoding='utf-8-sig')

    print('Export files:')
    print(final_path)
    print(clean_path)
    print(human_path)
    print(deny_path)
    print(failed_path)

## 18. Aggregate Analytics

Нийт үр дүнгийн summary. Энэ хэсэг нь competition-ийн aggregate reasoning шаардлагад нийцнэ.

In [ ]:
def count_flag(df: pd.DataFrame, flag: str) -> int:
    if df.empty or 'risk_flags' not in df.columns:
        return 0
    return df['risk_flags'].astype(str).str.contains(flag, regex=False).sum()

if results_df.empty:
    print('Summary гаргах result алга.')
else:
    summary = {
        'total_files': len(results_df),
        'successfully_processed': int((results_df['processing_status'] == 'SUCCESS').sum()) if 'processing_status' in results_df else 0,
        'failed_files': int((results_df['processing_status'] == 'FAILED').sum()) if 'processing_status' in results_df else 0,
        'auto_post': int((results_df['final_decision'] == 'AUTO_POST').sum()) if 'final_decision' in results_df else 0,
        'human_approval': int((results_df['final_decision'] == 'HUMAN_APPROVAL').sum()) if 'final_decision' in results_df else 0,
        'deny': int((results_df['final_decision'] == 'DENY').sum()) if 'final_decision' in results_df else 0,
        'amount_mismatch': int(count_flag(results_df, 'AMOUNT_MISMATCH')),
        'unregistered_vendor': int(count_flag(results_df, 'UNREGISTERED_VENDOR')),
        'invalid_date': int(count_flag(results_df, 'INVALID_DATE')),
        'bank_account_mismatch': int(count_flag(results_df, 'BANK_ACCOUNT_MISMATCH')),
        'duplicate': int(count_flag(results_df, 'DUPLICATE')),
    }

    summary_df = pd.DataFrame([summary])
    display(summary_df)

    print('========== PROJECT SUMMARY ==========', )
    print(f"Нийт боловсруулсан файл: {summary['total_files']}")
    print(f"AUTO_POST: {summary['auto_post']}")
    print(f"HUMAN_APPROVAL: {summary['human_approval']}")
    print(f"DENY: {summary['deny']}")
    print(f"Бүртгэлгүй vendor: {summary['unregistered_vendor']}")
    print(f"Duplicate invoice: {summary['duplicate']}")
    print('====================================')

## 19. Chatbot Q&A Agent

Invoice боловсруулсны дараа generated result table нь chatbot-ийн knowledge base болно. Энэ chatbot нь aggregate analytical асуултад хариулна.

In [ ]:
def list_rows_by_condition(df: pd.DataFrame, condition, columns=None, limit=10) -> str:
    if df.empty:
        return 'Result table хоосон байна.'
    filtered = df[condition]
    if filtered.empty:
        return 'Ийм нөхцөлтэй invoice олдсонгүй.'
    if columns is None:
        columns = ['file_name', 'vendor_name', 'total_amount', 'category', 'risk_flags', 'final_decision']
    available = [c for c in columns if c in filtered.columns]
    return filtered[available].head(limit).to_string(index=False)


def answer_question(question: str, df: pd.DataFrame) -> str:
    """Rule-based analytical chatbot over invoice results."""
    if df.empty:
        return 'Одоогоор боловсруулсан invoice result байхгүй байна.'

    q = normalize_text(question)

    if any(x in q for x in ['хэдэн invoice', 'нийт invoice', 'total invoice', 'how many invoices']):
        return f'Нийт боловсруулсан invoice файлын тоо: {len(df)}'

    if 'deny' in q or 'татгалз' in q:
        count = int((df['final_decision'] == 'DENY').sum()) if 'final_decision' in df else 0
        if any(x in q for x in ['харуул', 'list', 'show', 'ямар']):
            return list_rows_by_condition(df, df['final_decision'] == 'DENY')
        return f'DENY болсон invoice-ийн тоо: {count}'

    if 'auto' in q or 'шууд' in q or 'бүртгэх' in q:
        count = int((df['final_decision'] == 'AUTO_POST').sum()) if 'final_decision' in df else 0
        return f'AUTO_POST буюу шууд бүртгэх invoice-ийн тоо: {count}'

    if 'human' in q or 'approval' in q or 'зөвшөөрөл' in q or 'хүн' in q:
        count = int((df['final_decision'] == 'HUMAN_APPROVAL').sum()) if 'final_decision' in df else 0
        if any(x in q for x in ['харуул', 'list', 'show', 'ямар']):
            return list_rows_by_condition(df, df['final_decision'] == 'HUMAN_APPROVAL')
        return f'HUMAN_APPROVAL шаардлагатай invoice-ийн тоо: {count}'

    if 'бүртгэлгүй' in q or 'unregistered' in q or 'vendor' in q and 'not' in q:
        count = count_flag(df, 'UNREGISTERED_VENDOR')
        if any(x in q for x in ['харуул', 'list', 'show', 'ямар']):
            return list_rows_by_condition(df, df['risk_flags'].astype(str).str.contains('UNREGISTERED_VENDOR', regex=False))
        return f'Бүртгэлгүй vendor-той invoice-ийн тоо: {count}'

    if 'duplicate' in q or 'давхард' in q:
        count = count_flag(df, 'DUPLICATE')
        return f'Duplicate invoice-ийн тоо: {count}'

    if 'amount' in q or 'дүн' in q or 'mismatch' in q:
        count = count_flag(df, 'AMOUNT_MISMATCH')
        return f'AMOUNT_MISMATCH буюу дүнгийн зөрүүтэй invoice-ийн тоо: {count}'

    if 'bank' in q or 'данс' in q:
        count = count_flag(df, 'BANK_ACCOUNT_MISMATCH')
        return f'BANK_ACCOUNT_MISMATCH буюу банкны дансны зөрүүтэй invoice-ийн тоо: {count}'

    if 'date' in q or 'огноо' in q:
        count = count_flag(df, 'INVALID_DATE')
        return f'INVALID_DATE буюу огнооны алдаатай invoice-ийн тоо: {count}'

    if 'category' in q or 'ангилал' in q:
        if 'category' not in df.columns:
            return 'Category column олдсонгүй.'
        return df['category'].value_counts().to_string()

    if 'нийт дүн' in q or 'total amount' in q or 'sum' in q:
        if 'total_amount' not in df.columns:
            return 'total_amount column олдсонгүй.'
        total = pd.to_numeric(df['total_amount'], errors='coerce').fillna(0).sum()
        return f'Боловсруулсан invoice-уудын нийт дүн: {total:,.2f}'

    return 'Энэ асуултад одоогоор chatbot-ийн predefined analytical logic-оор хариулах боломжгүй байна. DENY, vendor, duplicate, amount, bank, date, category, total amount талаар асуугаарай.'

# Demo questions
sample_questions = [
    'Хэдэн invoice DENY болсон бэ?',
    'Бүртгэлгүй vendor хэд байна?',
    'Duplicate invoice байна уу?',
    'Ямар invoice хүний зөвшөөрөл шаардаж байна?',
    'Ангилал тус бүрийн тоо хэд вэ?',
    'Нийт дүн хэд вэ?',
]

for q in sample_questions:
    print('\nQ:', q)
    print('A:', answer_question(q, results_df))

## 20. Optional Gradio Interface

Энэ cell нь demo хийхэд ашиглаж болох interface үүсгэнэ. Kaggle дээр public URL ажиллах эсэх нь runtime орчноос шалтгаална. Notebook-ийн үндсэн pipeline interface-гүй ч бүрэн ажиллана.

In [ ]:
ENABLE_GRADIO_DEMO = False  # Interface ажиллуулах бол True болгоно.

if ENABLE_GRADIO_DEMO:
    import gradio as gr

    chatbot_df = results_df.copy()

    def gradio_ask(question):
        return answer_question(question, chatbot_df)

    def gradio_process_file(file):
        if file is None:
            return 'Файл upload хийгээгүй байна.', None
        path = Path(file.name)
        result = process_single_invoice(path)
        df = pd.DataFrame([result])
        return result.get('explanation', ''), df

    with gr.Blocks(title='Invoice Automation AI Agent') as demo:
        gr.Markdown('# Invoice Automation AI Agent')
        gr.Markdown('PDF/JPG/PNG invoice upload хийж, validation result болон chatbot Q&A ашиглана.')

        with gr.Tab('Single Invoice Demo'):
            file_input = gr.File(label='Upload invoice file')
            process_btn = gr.Button('Process Invoice')
            explanation_output = gr.Textbox(label='Explanation')
            table_output = gr.Dataframe(label='Result')
            process_btn.click(gradio_process_file, inputs=file_input, outputs=[explanation_output, table_output])

        with gr.Tab('Chatbot Q&A'):
            question = gr.Textbox(label='Ask a question', placeholder='Хэдэн invoice DENY болсон бэ?')
            answer = gr.Textbox(label='Answer')
            ask_btn = gr.Button('Ask')
            ask_btn.click(gradio_ask, inputs=question, outputs=answer)

    demo.launch(share=True)
else:
    print('Gradio demo disabled. ENABLE_GRADIO_DEMO=True болгож ажиллуулж болно.')

## 21. Limitations and Future Improvements

**Limitations:**

- Extraction accuracy depends on invoice image quality and Vision model response.
- Bank account validation works only if master database contains bank account columns.
- Category classification uses historical matching and keyword rules, not a trained classifier.
- Chatbot Q&A is analytical and rule-based; it is designed for reliable numeric answers.

**Future improvements:**

- Add OCR + layout parser fallback for API-free mode.
- Improve duplicate detection using embedding similarity.
- Add confidence scoring for extracted fields.
- Expand chatbot with LLM-based natural language interpretation.
- Deploy full Streamlit/Gradio web app.